# Online Retail — Customer Segmentation (Refitted)
### Country Segregation · ARIMA · SARIMA · KMeans · ML Classifiers
**Run cells in order. Upload `OnlineRetail.csv` when prompted in Cell 3.**

---
**What this notebook covers:**
1. Data cleaning & validation
2. EDA — revenue, orders, product & country insights
3. **Country segregation** — profile all countries, assign Revenue Tiers
4. Select **5 focus countries** for time-series deep-dive
5. **Stationarity tests (ADF)** per country
6. **ACF/PACF** analysis → ARIMA order selection
7. **ARIMA** — auto order via AIC grid search per country
8. **SARIMA(1,1,1)(1,1,0,12)** — seasonal monthly forecasting per country
9. **12-month forecast** per country with 95% confidence intervals
10. **RFM feature engineering** — global + per country breakdown
11. **KMeans clustering** — elbow + silhouette → 4 segments
12. **4 ML classifiers** — Logistic Regression, Decision Tree, Random Forest, GBM
13. Hyperparameter tuning · Feature importance · Business insights
14. Save model pipeline & download


## CELL 1 — Install Libraries

In [ ]:
# !pip install statsmodels --upgrade -q
# !pip install xgboost --quiet

## CELL 2 — Import All Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import itertools
import joblib
import json
import os

# ── Time Series ─────────────────────────────────────────────
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ── Sklearn ──────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (
    train_test_split, GridSearchCV,
    cross_val_score, StratifiedKFold
)
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.decomposition import PCA

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = ["#3B82F6", "#22C55E", "#F97316", "#8B5CF6", "#EF4444",
           "#0EA5A0", "#FBBF24", "#EC4899", "#14B8A6", "#6366F1"]

print("✅ All libraries imported successfully")

## CELL 3 — Upload & Load Dataset

In [ ]:
# ─────────────────────────────────────────────────────────
# DATASET LOADING — choose ONE option below
# ─────────────────────────────────────────────────────────

# OPTION A — Upload from your computer (recommended for Colab)
from google.colab import files
print("📂 Please upload OnlineRetail.csv when the dialog appears...")
uploaded = files.upload()
csv_name = [k for k in uploaded.keys() if k.endswith('.csv')][0]
df_raw = pd.read_csv(csv_name, encoding="latin-1")

# OPTION B — Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# df_raw = pd.read_csv('/content/drive/MyDrive/OnlineRetail.csv', encoding="latin-1")

# OPTION C — Direct path (if running locally)
# df_raw = pd.read_csv("OnlineRetail.csv", encoding="latin-1")

print(f"\n✅ Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"   Columns: {list(df_raw.columns)}")
df_raw.head()


## CELL 4 — Data Cleaning & Validation

In [ ]:
print("=" * 60)
print("DATA CLEANING & VALIDATION")
print("=" * 60)

df = df_raw.copy()

# Standardise column names (handle both naming conventions)
df.rename(columns={
    "InvoiceNo": "Invoice",
    "UnitPrice":  "Price",
}, inplace=True)

# Parse InvoiceDate
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print(f"\n  Raw shape   : {df.shape}")
print(f"  Date range  : {df['InvoiceDate'].min().date()}  →  {df['InvoiceDate'].max().date()}")
print(f"\n  Missing values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
for col in df.columns:
    if missing[col] > 0:
        print(f"    {col:<20}: {missing[col]:>6,}  ({missing_pct[col]:.2f}%)")

# [1] Drop rows with missing CustomerID
before = len(df)
df.dropna(subset=["CustomerID"], inplace=True)
print(f"\n  [1] Dropped {before - len(df):,} rows — missing CustomerID")

# [2] Fill missing Description
df["Description"].fillna("Unknown", inplace=True)

# [3] Remove cancelled orders (Invoice starts with 'C')
before = len(df)
df = df[~df["Invoice"].astype(str).str.startswith("C")]
print(f"  [2] Removed {before - len(df):,} cancelled invoices")

# [4] Remove rows with Quantity ≤ 0 or Price ≤ 0
before = len(df)
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
print(f"  [3] Removed {before - len(df):,} rows with Quantity/Price ≤ 0")

# [5] Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"  [4] Removed {before - len(df):,} duplicate rows")

# [6] Compute TotalAmount
df["TotalAmount"] = (df["Quantity"] * df["Price"]).round(2)

# [7] Extract date parts
df["Year"]       = df["InvoiceDate"].dt.year
df["Month"]      = df["InvoiceDate"].dt.to_period("M")
df["DayOfWeek"]  = df["InvoiceDate"].dt.day_name()
df["Hour"]       = df["InvoiceDate"].dt.hour

# CustomerID → string for consistency
df["CustomerID"] = df["CustomerID"].astype(str).str.strip()

print(f"\n  ✅ Clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Unique customers  : {df['CustomerID'].nunique():,}")
print(f"  Unique products   : {df['StockCode'].nunique():,}")
print(f"  Unique countries  : {df['Country'].nunique():,}")
print(f"  Unique invoices   : {df['Invoice'].nunique():,}")
df.head()

## CELL 5 — EDA: Revenue & Business Insights

In [ ]:
print("=" * 60)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Monthly Revenue Trend
monthly_rev = (df.groupby("Month")["TotalAmount"]
               .sum()
               .reset_index())
monthly_rev["Month_str"] = monthly_rev["Month"].astype(str)
axes[0,0].bar(monthly_rev["Month_str"], monthly_rev["TotalAmount"]/1e3,
              color=PALETTE[0], alpha=0.85)
axes[0,0].set_title("Monthly Revenue Trend (£K)")
axes[0,0].set_xlabel("Month"); axes[0,0].set_ylabel("Revenue (£K)")
axes[0,0].tick_params(axis="x", rotation=45)
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"£{x:.0f}K"))

# 2. Top 10 Countries by Revenue
top_countries_rev = (df.groupby("Country")["TotalAmount"]
                     .sum().nlargest(10).sort_values())
axes[0,1].barh(top_countries_rev.index,
               top_countries_rev.values/1e3,
               color=PALETTE[1], alpha=0.85)
axes[0,1].set_title("Top 10 Countries by Revenue (£K)")
axes[0,1].set_xlabel("Revenue (£K)")
axes[0,1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"£{x:.0f}K"))

# 3. Orders by Day of Week
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
dow_counts = df.groupby("DayOfWeek")["Invoice"].nunique().reindex(dow_order)
axes[1,0].bar(dow_counts.index, dow_counts.values, color=PALETTE[2], alpha=0.85)
axes[1,0].set_title("Orders by Day of Week")
axes[1,0].set_xlabel("Day"); axes[1,0].set_ylabel("Unique Orders")
axes[1,0].tick_params(axis="x", rotation=30)

# 4. Orders by Hour
hourly = df.groupby("Hour")["Invoice"].nunique()
axes[1,1].plot(hourly.index, hourly.values,
               color=PALETTE[3], lw=2.2, marker="o", ms=4)
axes[1,1].fill_between(hourly.index, hourly.values, alpha=0.12, color=PALETTE[3])
axes[1,1].set_title("Orders by Hour of Day (Peak: 10AM–2PM)")
axes[1,1].set_xlabel("Hour"); axes[1,1].set_ylabel("Unique Orders")

plt.suptitle("Online Retail — Exploratory Data Analysis",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

# Top 10 Products
print("\nTop 10 Products by Revenue:")
top_products = (df.groupby("Description")["TotalAmount"]
                .sum().nlargest(10)
                .reset_index()
                .rename(columns={"TotalAmount":"Revenue £"}))
top_products["Revenue £"] = top_products["Revenue £"].map("£{:,.2f}".format)
display(top_products)

## CELL 6 — COUNTRY SEGREGATION (All Countries)

In [ ]:
print("=" * 60)
print("COUNTRY SEGREGATION — Profiling All Countries")
print("=" * 60)

country_profile = df.groupby("Country").agg(
    Total_Revenue  = ("TotalAmount", "sum"),
    Avg_Order_Val  = ("TotalAmount", "mean"),
    Unique_Customers = ("CustomerID","nunique"),
    Unique_Orders  = ("Invoice",    "nunique"),
    Unique_Products= ("StockCode",  "nunique"),
    Total_Quantity = ("Quantity",   "sum"),
).round(2).reset_index()

country_profile["Revenue_Share_%"] = (
    country_profile["Total_Revenue"] /
    country_profile["Total_Revenue"].sum() * 100
).round(2)

country_profile["Revenue_Tier"] = pd.qcut(
    country_profile["Total_Revenue"],
    q=3, labels=["Low","Mid","High"]
)

country_profile = country_profile.sort_values("Total_Revenue", ascending=False)

print(f"\n  Total countries: {len(country_profile)}")
print(f"\n  Revenue Tiers:")
for tier in ["High","Mid","Low"]:
    ctries = country_profile[country_profile["Revenue_Tier"]==tier]["Country"].tolist()
    print(f"    {tier:4s}: {ctries}")

print(f"\n  Top 10 countries by revenue:")
top10 = country_profile.head(10)[
    ["Country","Total_Revenue","Revenue_Share_%","Unique_Customers","Revenue_Tier"]
].copy()
top10["Total_Revenue"] = top10["Total_Revenue"].map("£{:,.2f}".format)
display(top10)

# Plot all countries heatmap style
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue share bar
cp_plot = country_profile.head(15).sort_values("Total_Revenue")
colors_bar = ["#22C55E" if t=="High" else "#3B82F6" if t=="Mid" else "#F97316"
              for t in cp_plot["Revenue_Tier"]]
axes[0].barh(cp_plot["Country"], cp_plot["Total_Revenue"]/1e3, color=colors_bar)
axes[0].set_title("Top 15 Countries — Total Revenue\n(Green=High, Blue=Mid, Orange=Low)")
axes[0].set_xlabel("Revenue (£K)")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"£{x:.0f}K"))

# Customer count
cp_cust = country_profile.nlargest(15,"Unique_Customers").sort_values("Unique_Customers")
axes[1].barh(cp_cust["Country"], cp_cust["Unique_Customers"], color=PALETTE[4], alpha=0.85)
axes[1].set_title("Top 15 Countries — Unique Customers")
axes[1].set_xlabel("Unique Customers")

plt.suptitle("Country Segregation — All Countries Profiled",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

## CELL 7 — Select 5 Focus Countries

In [ ]:
# Pick top-1 (UK dominates), then top 4 international by revenue
# UK is excluded from time-series due to scale dominance
non_uk = country_profile[country_profile["Country"] != "United Kingdom"]

FOCUS_COUNTRIES = (
    ["United Kingdom"] +
    non_uk.nlargest(4, "Total_Revenue")["Country"].tolist()
)

COUNTRY_COLORS = {
    FOCUS_COUNTRIES[0]: "#2ECC71",
    FOCUS_COUNTRIES[1]: "#3B82F6",
    FOCUS_COUNTRIES[2]: "#F97316",
    FOCUS_COUNTRIES[3]: "#8B5CF6",
    FOCUS_COUNTRIES[4]: "#EF4444",
}

print("=" * 60)
print("FOCUS COUNTRIES SELECTED FOR DEEP-DIVE ANALYSIS")
print("=" * 60)
for c in FOCUS_COUNTRIES:
    row = country_profile[country_profile["Country"]==c].iloc[0]
    print(f"  {c:<25} | Tier: {row['Revenue_Tier']:4s} | "
          f"Revenue: £{row['Total_Revenue']:>12,.2f} | "
          f"Customers: {row['Unique_Customers']:>5,} | "
          f"Share: {row['Revenue_Share_%']:.2f}%")

# Historical monthly revenue overlay for 5 focus countries
fig, ax = plt.subplots(figsize=(14, 5))
for country in FOCUS_COUNTRIES:
    ts = (df[df["Country"]==country]
          .groupby("Month")["TotalAmount"]
          .sum()
          .reset_index()
          .sort_values("Month"))
    ts["Month_dt"] = ts["Month"].dt.to_timestamp()
    ax.plot(ts["Month_dt"], ts["TotalAmount"]/1e3,
            lw=1.8, marker="o", ms=3,
            label=country, color=COUNTRY_COLORS[country])

ax.set_title("Monthly Revenue — 5 Focus Countries",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Month"); ax.set_ylabel("Revenue (£K)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"£{x:.0f}K"))
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

## CELL 8 — Stationarity Tests (ADF) per Country

In [ ]:
print("=" * 60)
print("AUGMENTED DICKEY-FULLER STATIONARITY TEST")
print("H0: Series has unit root (non-stationary)")
print("Reject H0 if p < 0.05  →  Series IS stationary")
print("=" * 60)

adf_results = {}
for country in FOCUS_COUNTRIES:
    ts = (df[df["Country"]==country]
          .groupby("Month")["TotalAmount"]
          .sum()
          .sort_index())
    ts.index = ts.index.to_timestamp()
    if len(ts) < 6:
        print(f"\n  {country:<25} — skipped (< 6 observations)")
        continue
    stat, p, _, _, crit, _ = adfuller(ts)
    status = "✅ STATIONARY" if p < 0.05 else "⚠️  NON-STATIONARY (d=1 needed)"
    adf_results[country] = {"stat": stat, "p": p, "stationary": p < 0.05}
    print(f"\n  {country:<25}")
    print(f"    ADF Statistic : {stat:.4f}")
    print(f"    p-value       : {p:.6f}")
    print(f"    Critical 5%   : {crit['5%']:.4f}")
    print(f"    Result        : {status}")

## CELL 9 — ACF & PACF Plots

In [ ]:
# Use country with the most data points (usually UK)
ref_country = FOCUS_COUNTRIES[0]
ts_ref = (df[df["Country"]==ref_country]
          .groupby("Month")["TotalAmount"]
          .sum()
          .sort_index())
ts_ref.index = ts_ref.index.to_timestamp()

n_lags = min(12, len(ts_ref)//2 - 1)

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
plot_acf(ts_ref, lags=n_lags, ax=axes[0],
         title=f"ACF — {ref_country} Monthly Revenue (lags={n_lags})")
plot_pacf(ts_ref, lags=n_lags, ax=axes[1],
          title=f"PACF — {ref_country} Monthly Revenue",
          method="ywm")
plt.suptitle("Autocorrelation Analysis — Guides ARIMA Order Selection",
             fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

print(f"\nSeries length for {ref_country}: {len(ts_ref)} months")
print("ACF / PACF help identify p and q for ARIMA.")
print("Significant spike at lag-1 in PACF → AR(1) term suitable.")

## CELL 10 — ARIMA Fitting (Auto Order via AIC)

In [ ]:
def fit_best_arima(ts, max_p=3, max_q=3):
    """Grid-search ARIMA(p,1,q) and return best by AIC."""
    best_aic, best_order, best_fit = np.inf, (1,1,1), None
    for p, q in itertools.product(range(max_p+1), range(max_q+1)):
        try:
            m = ARIMA(ts, order=(p,1,q)).fit()
            if m.aic < best_aic:
                best_aic, best_order, best_fit = m.aic, (p,1,q), m
        except Exception:
            pass
    return best_fit, best_order, best_aic

print("=" * 60)
print("ARIMA — Auto Order Selection via AIC Grid Search")
print("Testing p ∈ {0..3}, d=1 (differencing), q ∈ {0..3}")
print("=" * 60)

arima_results = {}
for country in FOCUS_COUNTRIES:
    ts = (df[df["Country"]==country]
          .groupby("Month")["TotalAmount"]
          .sum()
          .sort_index())
    ts.index = ts.index.to_timestamp()
    ts.index.freq = "MS"

    if len(ts) < 8:
        print(f"\n  {country:<25} — skipped (< 8 observations)")
        continue

    n = len(ts)
    split = max(int(n * 0.80), n - 3)
    train = ts.iloc[:split]
    test  = ts.iloc[split:]

    fit, order, aic = fit_best_arima(train, max_p=3, max_q=3)
    fc = fit.forecast(steps=len(test))

    # metrics already imported at top
    # from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score as r2
    rmse = np.sqrt(mean_squared_error(test.values, fc.values))
    mae  = mean_absolute_error(test.values, fc.values)
    r2_val = r2(test.values, fc.values) if len(test) > 1 else float("nan")

    arima_results[country] = {
        "order": order, "aic": round(aic,2),
        "rmse": round(rmse,2), "mae": round(mae,2), "r2": round(r2_val,4),
        "fit": fit, "train": train, "test": test, "fc": fc
    }
    tier = country_profile[country_profile["Country"]==country]["Revenue_Tier"].values[0]
    print(f"\n  {country:<25} ({tier}) → Best ARIMA{order}")
    print(f"    AIC  : {aic:.2f}")
    print(f"    RMSE : £{rmse:>10,.2f}")
    print(f"    MAE  : £{mae:>10,.2f}")
    print(f"    R²   : {r2_val:.4f}")

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## CELL 11 — SARIMA Fitting for All 5 Focus Countries

In [ ]:
print("=" * 60)
print("SARIMA — Fitting per Country")
print("Order (1,1,1) | Seasonal (1,D,0,12) — D auto-detected")
print("s=12 captures annual monthly seasonality")
print("=" * 60)

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def fit_sarima_safe(train_ts, test_ts):
    """
    Fit SARIMA with automatic seasonal differencing detection.
    Falls back gracefully when series is too short for (1,1,0,12).
    """
    n_train = len(train_ts)

    # Try orders from most to least complex
    orders_to_try = [
        ((1,1,1), (1,1,0,12), "SARIMA(1,1,1)(1,1,0,12)"),
        ((1,1,1), (1,0,0,12), "SARIMA(1,1,1)(1,0,0,12)"),
        ((1,1,1), (0,0,0, 0), "ARIMA(1,1,1) fallback"),
    ]

    for order, seasonal_order, label in orders_to_try:
        # Need at least s*(D+1)+p+d observations for seasonal differencing
        s = seasonal_order[3] if seasonal_order[3] > 0 else 1
        D = seasonal_order[1]
        min_needed = s * (D + 1) + order[0] + order[1] + 4
        if n_train < min_needed:
            print(f"    ↳ {label} needs {min_needed} obs, have {n_train} — skipping")
            continue
        try:
            if seasonal_order[3] > 0:
                fit = SARIMAX(train_ts,
                              order=order,
                              seasonal_order=seasonal_order,
                              enforce_stationarity=False,
                              enforce_invertibility=False).fit(disp=False)
            else:
                fit = ARIMA(train_ts, order=order).fit()
            return fit, label
        except Exception as e:
            print(f"    ↳ {label} failed: {e}")
            continue

    return None, "none"

sarima_results = {}
for country in FOCUS_COUNTRIES:
    ts = (df[df["Country"]==country]
          .groupby("Month")["TotalAmount"]
          .sum()
          .sort_index())
    ts.index = ts.index.to_timestamp()
    ts.index.freq = "MS"

    if len(ts) < 6:
        print(f"\n  {country:<25} — skipped (only {len(ts)} months)")
        continue

    n = len(ts)
    # Use at least 3 test points, but no more than 20% of data
    split = max(n - 4, int(n * 0.80))
    split = min(split, n - 2)   # always keep ≥ 2 test points
    train = ts.iloc[:split]
    test  = ts.iloc[split:]

    fit, model_label = fit_sarima_safe(train, test)

    if fit is None:
        print(f"\n  {country:<25} — all SARIMA attempts failed, skipping")
        continue

    fc   = fit.forecast(steps=len(test))
    rmse = np.sqrt(mean_squared_error(test.values, fc.values))
    mae  = mean_absolute_error(test.values, fc.values)
    r2_val = r2_score(test.values, fc.values) if len(test) > 1 else float("nan")

    sarima_results[country] = {
        "model_label": model_label,
        "aic":  round(fit.aic, 2),
        "rmse": round(rmse, 2),
        "mae":  round(mae, 2),
        "r2":   round(r2_val, 4),
        "fit":  fit,
        "train": train,
        "test":  test,
        "fc":    fc,
    }
    tier = country_profile[country_profile["Country"]==country]["Revenue_Tier"].values[0]
    print(f"\n  {country:<25} ({tier})")
    print(f"    Model : {model_label}")
    print(f"    Series: {len(ts)} months (train={len(train)}, test={len(test)})")
    print(f"    AIC   : {fit.aic:.2f}")
    print(f"    RMSE  : £{rmse:>10,.2f}")
    print(f"    MAE   : £{mae:>10,.2f}")
    print(f"    R²    : {r2_val:.4f}")

print(f"\n✅  SARIMA fitted for {len(sarima_results)} countries")


## CELL 12 — ARIMA vs SARIMA Comparison

In [ ]:
# Guard: rebuild common list from what actually fitted
common = [c for c in FOCUS_COUNTRIES
          if c in arima_results and c in sarima_results]
print(f"Countries with both ARIMA & SARIMA results: {common}")

if not common:
    print("⚠️  No common results — check that Cells 10 and 11 ran successfully.")
else:
    common = [c for c in FOCUS_COUNTRIES
              if c in arima_results and c in sarima_results]
    
    arima_rmse_vals  = [arima_results[c]["rmse"]/1e3  for c in common]
    sarima_rmse_vals = [sarima_results[c]["rmse"]/1e3 for c in common]
    arima_r2_vals    = [arima_results[c]["r2"]        for c in common]
    sarima_r2_vals   = [sarima_results[c]["r2"]       for c in common]
    short_labels     = [c.replace("United Kingdom","UK")
                         .replace("Netherlands","NL")
                         .replace("Germany","DE")
                         .replace("France","FR")
                         .replace("Australia","AU") for c in common]
    
    x = np.arange(len(common)); w = 0.35
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    axes[0].bar(x-w/2, arima_rmse_vals,  w, label="ARIMA",  color="#94A3B8", alpha=0.9)
    axes[0].bar(x+w/2, sarima_rmse_vals, w, label="SARIMA", color="#22C55E", alpha=0.9)
    axes[0].set_xticks(x); axes[0].set_xticklabels(short_labels)
    axes[0].set_title("RMSE Comparison (K£)\nLower = Better"); axes[0].legend()
    
    axes[1].bar(x-w/2, arima_r2_vals,  w, label="ARIMA",  color="#94A3B8", alpha=0.9)
    axes[1].bar(x+w/2, sarima_r2_vals, w, label="SARIMA", color="#3B82F6", alpha=0.9)
    axes[1].set_xticks(x); axes[1].set_xticklabels(short_labels)
    axes[1].set_title("R² Score Comparison\nHigher = Better"); axes[1].legend()
    
    sarima_wins = sum(1 for c in common
                      if sarima_results[c]["rmse"] < arima_results[c]["rmse"])
    axes[2].bar(["ARIMA","SARIMA"],
                [len(common)-sarima_wins, sarima_wins],
                color=["#94A3B8","#22C55E"], alpha=0.9, width=0.4)
    axes[2].set_title(f"Model Win Count\nSARIMA wins {sarima_wins}/{len(common)} countries")
    for i, v in enumerate([len(common)-sarima_wins, sarima_wins]):
        axes[2].text(i, v+0.05, str(v), ha="center", fontsize=14, fontweight="bold")
    
    plt.suptitle("ARIMA vs SARIMA — Revenue Forecast Comparison (5 Countries)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()
    
    print("\n" + "=" * 72)
    print(f"  {'Country':<26}| {'ARIMA Order':>12} | {'ARIMA R²':>9} | {'SARIMA R²':>9} | {'SARIMA RMSE £':>13} | Winner")
    print("  " + "-"*72)
    for c in common:
        a  = arima_results[c]
        s  = sarima_results[c]
        tier = country_profile[country_profile["Country"]==c]["Revenue_Tier"].values[0]
        winner = "SARIMA ✓" if s["rmse"] < a["rmse"] else "ARIMA  ✓"
        print(f"  {c:<26}| {str(a['order']):>12} | {a['r2']:>9.4f} | {s['r2']:>9.4f} | £{s['rmse']:>11,.2f} | {winner}")

## CELL 13 — SARIMA Test-Set Fit Plots (5 Countries)

In [ ]:
fig, axes = plt.subplots(len(common), 1,
                          figsize=(14, 4*len(common)), sharex=False)
if len(common) == 1:
    axes = [axes]

fig.suptitle("SARIMA — Test Set Fit vs Actual Revenue",
             fontsize=14, fontweight="bold", y=1.01)

for idx, country in enumerate(common):
    s  = sarima_results[country]
    ax = axes[idx]
    color = COUNTRY_COLORS.get(country, PALETTE[idx % len(PALETTE)])
    tier  = country_profile[country_profile["Country"]==country]["Revenue_Tier"].values[0]

    ax.plot(s["train"].index, s["train"].values/1e3,
            color="steelblue", lw=1.5, alpha=0.7, label="Train")
    ax.plot(s["test"].index,  s["test"].values/1e3,
            color="black",     lw=2.0, label="Actual (Test)")
    ax.plot(s["fc"].index,    s["fc"].values/1e3,
            color=color, lw=2.2, linestyle="--", marker="o", ms=4,
            label=f"SARIMA Forecast (R²={s['r2']:.3f})")
    ax.axvline(s["train"].index[-1], color="gray", lw=1, linestyle=":")
    ax.set_title(f"{country} ({tier}) | RMSE=£{s['rmse']:,.0f} | MAE=£{s['mae']:,.0f}",
                 fontsize=11)
    ax.set_ylabel("Revenue (£K)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"£{x:.0f}K"))
    ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

## CELL 14 — 12-Month Forecast per Country (SARIMA)

In [ ]:
print("=" * 65)
print("SARIMA 12-MONTH FORECAST — Refit on Full Data per Country")
print("=" * 65)

FORECAST_MONTHS = 12
country_forecasts = {}
common = [c for c in FOCUS_COUNTRIES if c in sarima_results]

if not common:
    print("⚠️  No SARIMA results found — re-run Cell 11 first.")
else:
    fig, axes = plt.subplots(len(common), 1,
                              figsize=(14, 5*len(common)), sharex=False)
    if len(common) == 1:
        axes = [axes]

    fig.suptitle("SARIMA 12-Month Ahead Forecast — Focus Countries\n"
                 "(Shaded = 95% Confidence Interval)",
                 fontsize=14, fontweight="bold", y=1.01)

    for idx, country in enumerate(common):
        ts = (df[df["Country"]==country]
              .groupby("Month")["TotalAmount"]
              .sum()
              .sort_index())
        ts.index = ts.index.to_timestamp()
        ts.index.freq = "MS"

        color = COUNTRY_COLORS.get(country, PALETTE[idx % len(PALETTE)])
        tier  = country_profile[country_profile["Country"]==country]["Revenue_Tier"].values[0]
        model_label = sarima_results[country]["model_label"]

        # Refit on FULL series (try orders from most to least complex)
        fit_full = None
        for s_ord, label in [((1,1,0,12),"SARIMA(1,1,1)(1,1,0,12)"),
                              ((1,0,0,12),"SARIMA(1,1,1)(1,0,0,12)"),
                              ((0,0,0, 0),"ARIMA(1,1,1) fallback")]:
            min_n = (s_ord[3]*(s_ord[1]+1)+4) if s_ord[3]>0 else 4
            if len(ts) < min_n:
                continue
            try:
                if s_ord[3] > 0:
                    fit_full = SARIMAX(ts,order=(1,1,1),seasonal_order=s_ord,
                                       enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
                else:
                    from statsmodels.tsa.arima.model import ARIMA as _ARIMA
                    fit_full = _ARIMA(ts, order=(1,1,1)).fit()
                break
            except Exception:
                continue
        if fit_full is None:
            print(f"  ⚠️  Could not refit on full data for {country}")
            continue

        try:
            fc_obj  = fit_full.get_forecast(steps=FORECAST_MONTHS)
            fc_mean = fc_obj.predicted_mean
            ci      = fc_obj.conf_int(alpha=0.05)
        except Exception:
            # Fallback: use simple forecast if get_forecast fails
            fc_mean = pd.Series(fit_full.forecast(steps=FORECAST_MONTHS))
            ci = pd.DataFrame({
                "lower": fc_mean * 0.85,
                "upper": fc_mean * 1.15,
            })

        country_forecasts[country] = {
            "dates":  [str(d.date()) if hasattr(d,"date") else str(d)
                       for d in (fc_mean.index.tolist()
                                 if hasattr(fc_mean.index,"tolist")
                                 else range(FORECAST_MONTHS))],
            "values": [round(float(v),2) for v in fc_mean.values],
            "lower":  [round(float(v),2) for v in ci.iloc[:,0].values],
            "upper":  [round(float(v),2) for v in ci.iloc[:,1].values],
        }

        # ── Print week-by-week table ─────────────────────────────────
        s = sarima_results[country]
        print(f"\n  {country} ({tier}) — {model_label} — 12-Month Forecast:")
        print(f"  {'Month':>6} | {'Date':>12} | {'Predicted £':>14} | {'Lower 95%':>12} | {'Upper 95%':>12}")
        print(f"  {'-'*63}")
        for i, (v, lo, hi) in enumerate(zip(
                country_forecasts[country]["values"],
                country_forecasts[country]["lower"],
                country_forecasts[country]["upper"])):
            date_str = country_forecasts[country]["dates"][i]
            print(f"  {i+1:>6} | {date_str:>12} | £{v:>13,.2f} | £{lo:>10,.2f} | £{hi:>10,.2f}")

        # ── Plot ─────────────────────────────────────────────────────
        ax = axes[idx]
        ax.plot(ts.index, ts.values/1e3, color="steelblue", lw=1.8,
                label="Historical Revenue")

        fc_dates = fc_mean.index if hasattr(fc_mean.index,"tolist") else pd.date_range(
            ts.index[-1], periods=FORECAST_MONTHS+1, freq="MS")[1:]
        ax.plot(fc_dates, fc_mean.values/1e3,
                color=color, lw=2.2, linestyle="--", marker="o", ms=5,
                label=f"12-Month Forecast")
        ax.fill_between(fc_dates,
                        ci.iloc[:,0].values/1e3, ci.iloc[:,1].values/1e3,
                        color=color, alpha=0.18, label="95% CI")
        ax.axvline(ts.index[-1], color="gray", lw=1.3, linestyle=":",
                   label="Forecast Start")
        avg_hist = ts.mean()
        ax.axhline(avg_hist/1e3, color="orange", lw=1,
                   linestyle="--", alpha=0.6,
                   label=f"Hist. Avg £{avg_hist/1e3:.1f}K")

        ax.set_title(
            f"{country} ({tier}) | {model_label} | "
            f"Test R²={s['r2']:.3f}  RMSE=£{s['rmse']:,.0f}",
            fontsize=11
        )
        ax.set_ylabel("Revenue (£K)")
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x,_: f"£{x:.0f}K"))
        ax.legend(fontsize=9, loc="upper left")

    plt.tight_layout()
    plt.show()
    print(f"\n✅  12-month forecasts generated for: {common}")


## CELL 15 — SARIMA Residual Diagnostics

In [ ]:
# ── SARIMA Residual Diagnostics ──────────────────────────────────────
# Safe wrapper: handles short series that crash plot_diagnostics

def safe_residual_diagnostics(country_name, fit_result):
    """Plot SARIMA residual diagnostics safely regardless of series length."""
    resid = fit_result.resid.dropna()
    n = len(resid)
    print(f"  Residuals length for {country_name}: {n} observations")

    if n < 5:
        print(f"  ⚠️  Too few residuals ({n}) — skipping diagnostics for {country_name}")
        return

    try:
        # Use lags = min(10, n//2 - 2) to avoid ValueError
        safe_lags = max(1, min(10, n // 2 - 2))

        if n >= 15:
            # statsmodels built-in diagnostic (passes lags argument safely)
            fig = fit_result.plot_diagnostics(
                figsize=(14, 8),
                lags=safe_lags
            )
            fig.suptitle(f"SARIMA Residual Diagnostics — {country_name}",
                         fontsize=12, fontweight="bold")
            plt.tight_layout()
            plt.show()
        else:
            # Manual fallback for very short series
            fig, axes = plt.subplots(2, 2, figsize=(13, 8))

            # 1. Residual time plot
            axes[0,0].plot(resid.index, resid.values, color="#3B82F6", lw=1.5)
            axes[0,0].axhline(0, color="red", linestyle="--", lw=1)
            axes[0,0].set_title("Residuals over Time")
            axes[0,0].set_ylabel("Residual")

            # 2. Histogram
            axes[0,1].hist(resid.values, bins=max(5, n//3),
                           color="#22C55E", edgecolor="white", alpha=0.8)
            axes[0,1].set_title("Residual Histogram (should be ~Normal)")
            axes[0,1].set_xlabel("Residual")

            # 3. Q-Q plot (manual)
            import scipy.stats as stats
            (osm, osr), (slope, intercept, r) = stats.probplot(resid.values)
            axes[1,0].plot(osm, osr, "o", color="#F97316", ms=5)
            axes[1,0].plot(osm, slope*np.array(osm)+intercept,
                           "r-", lw=1.5, label=f"R²={r**2:.3f}")
            axes[1,0].set_title("Q-Q Plot (points on line = Normal)")
            axes[1,0].legend()

            # 4. ACF of residuals
            from statsmodels.graphics.tsaplots import plot_acf
            plot_acf(resid, lags=safe_lags, ax=axes[1,1],
                     title=f"ACF of Residuals (no spikes = good fit)")

            plt.suptitle(f"SARIMA Residual Diagnostics — {country_name}",
                         fontsize=12, fontweight="bold")
            plt.tight_layout()
            plt.show()

    except Exception as e:
        print(f"  ⚠️  plot_diagnostics failed for {country_name}: {e}")
        print("  → Showing simple residual plot instead")
        resid = fit_result.resid.dropna()
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(resid.index, resid.values, color="#3B82F6", lw=1.5)
        ax.axhline(0, color="red", linestyle="--", lw=1)
        ax.set_title(f"SARIMA Residuals — {country_name}")
        ax.set_ylabel("Residual")
        plt.tight_layout()
        plt.show()

# Run diagnostics for each country in sarima_results
for country in sarima_results:
    safe_residual_diagnostics(country, sarima_results[country]["fit"])

print("\nInterpretation:")
print("  Residual plot : Should look random (no trend/seasonality left)")
print("  Histogram     : Should be roughly bell-shaped / Normal")
print("  Q-Q plot      : Points close to the line = Normal residuals = good SARIMA fit")
print("  ACF residuals : No significant spikes = residuals are white noise = good fit")


## CELL 16 — RFM Feature Engineering (Global + Per Country)

In [ ]:
print("=" * 60)
print("RFM FEATURE ENGINEERING")
print("=" * 60)

snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print(f"  Snapshot date: {snapshot_date.date()}")

# Global RFM
rfm_global = df.groupby("CustomerID").agg(
    Recency   = ("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency = ("Invoice",     "nunique"),
    Monetary  = ("TotalAmount", "sum"),
    Country   = ("Country",     "first"),
).reset_index()
rfm_global["AvgOrderValue"] = (rfm_global["Monetary"] / rfm_global["Frequency"]).round(2)

# IQR capping per feature
for col in ["Recency","Frequency","Monetary"]:
    q1, q3 = rfm_global[col].quantile(0.05), rfm_global[col].quantile(0.95)
    rfm_global[col] = rfm_global[col].clip(q1, q3)

print(f"\n  Global RFM: {rfm_global.shape[0]:,} customers")
display(rfm_global.describe().round(2))

# Per-country RFM summary
rfm_country = rfm_global.groupby("Country").agg(
    Customers  = ("CustomerID",   "count"),
    Avg_Recency= ("Recency",      "mean"),
    Avg_Freq   = ("Frequency",    "mean"),
    Avg_Monetary=("Monetary",     "mean"),
).round(2).reset_index()

print("\n  RFM by Country (Focus Countries):")
display(rfm_country[rfm_country["Country"].isin(FOCUS_COUNTRIES)])

## CELL 17 — RFM Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rfm_plot_cols = ["Recency", "Frequency", "Monetary"]
rfm_colors    = [PALETTE[0], PALETTE[1], PALETTE[2]]
rfm_labels    = ["Recency (days)", "Frequency (orders)", "Monetary (£)"]

for ax, col, color, lbl in zip(axes, rfm_plot_cols, rfm_colors, rfm_labels):
    ax.hist(rfm_global[col], bins=40, color=color,
            edgecolor="white", alpha=0.85)
    ax.set_title(f"{col} Distribution")
    ax.set_xlabel(lbl)
    ax.set_ylabel("Customers")
    mean_val = rfm_global[col].mean()
    ax.axvline(mean_val, color="red", linestyle="--", lw=1.5,
               label=f"Mean={mean_val:.1f}")
    ax.legend(fontsize=9)

plt.suptitle("RFM Feature Distributions — After Outlier Capping",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("\n  RFM Distribution Summary:")
display(rfm_global[["Recency","Frequency","Monetary"]].describe().round(2))


## CELL 18 — KMeans Clustering: Elbow + Silhouette

In [ ]:
print("=" * 60)
print("K-MEANS CUSTOMER SEGMENTATION")
print("=" * 60)

scaler_rfm = StandardScaler()
rfm_scaled = scaler_rfm.fit_transform(
    rfm_global[["Recency","Frequency","Monetary"]]
)

inertias, silhouettes = [], []
K_range = range(2, 11)

for k in K_range:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(rfm_scaled, labels,
                           sample_size=min(3000, len(rfm_global)),
                           random_state=42)
    silhouettes.append(sil)
    print(f"  k={k}: Inertia={km.inertia_:>10,.1f}  Silhouette={sil:.4f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(list(K_range), inertias, "o-", color=PALETTE[0], lw=2)
ax1.set_xlabel("k"); ax1.set_ylabel("Inertia (SSE)")
ax1.set_title("Elbow Method — Optimal k")

best_k = list(K_range)[silhouettes.index(max(silhouettes))]
ax2.plot(list(K_range), silhouettes, "o-", color=PALETTE[1], lw=2)
ax2.axvline(best_k, color="red", lw=1.5, linestyle="--",
            label=f"Best k={best_k}")
ax2.set_xlabel("k"); ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Score — Optimal k"); ax2.legend()

plt.suptitle("Finding Optimal Number of Clusters",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

# Use k=4 for business interpretability
BEST_K = 4
print(f"\n  Using k={BEST_K} (business-meaningful 4-segment RFM model)")

kmeans = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
rfm_global["Cluster"] = kmeans.fit_predict(rfm_scaled)

sil_final = silhouette_score(rfm_scaled, rfm_global["Cluster"],
                              sample_size=min(3000, len(rfm_global)),
                              random_state=42)
dbi_final = davies_bouldin_score(rfm_scaled, rfm_global["Cluster"])
print(f"  Silhouette Score : {sil_final:.4f}")
print(f"  Davies-Bouldin   : {dbi_final:.4f}")

## CELL 19 — Segment Labelling & Analysis

In [ ]:
cluster_summary = rfm_global.groupby("Cluster")[
    ["Recency","Frequency","Monetary","AvgOrderValue"]
].mean().round(2)
cluster_summary["Count"] = rfm_global.groupby("Cluster").size()
cluster_summary["Pct"]   = (
    cluster_summary["Count"] / len(rfm_global) * 100
).round(1)

print("Cluster Profiles (before labelling):")
display(cluster_summary)

# Auto-label: rank by (low Recency + high Freq + high Monetary)
cluster_summary["Score"] = (
    cluster_summary["Recency"].rank(ascending=True) * -1 +
    cluster_summary["Frequency"].rank(ascending=False) +
    cluster_summary["Monetary"].rank(ascending=False)
)
rank_order = cluster_summary["Score"].sort_values(ascending=False).index.tolist()
_label_map = {
    rank_order[0]: "Champions",
    rank_order[1]: "Loyal Customers",
    rank_order[2]: "At-Risk Customers",
    rank_order[3]: "New / Occasional",
}
SEG_COLORS = {
    "Champions":       "#22C55E",
    "Loyal Customers": "#3B82F6",
    "At-Risk Customers":"#EF4444",
    "New / Occasional":"#F97316",
}

rfm_global["Segment"]      = rfm_global["Cluster"].map(_label_map)
rfm_global["SegmentLabel"] = rfm_global["Segment"]

print("\nSegment Summary:")
seg_summary = rfm_global.groupby("Segment")[
    ["Recency","Frequency","Monetary","AvgOrderValue"]
].mean().round(2)
seg_summary["Count"] = rfm_global.groupby("Segment").size()
seg_summary["Pct_%"] = (seg_summary["Count"]/len(rfm_global)*100).round(1)
display(seg_summary)

## CELL 20 — Segment Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Pie chart
seg_counts = rfm_global["Segment"].value_counts()
colors_pie = [SEG_COLORS[s] for s in seg_counts.index]
axes[0,0].pie(seg_counts.values, labels=seg_counts.index,
              colors=colors_pie, autopct="%1.1f%%", startangle=140,
              wedgeprops={"edgecolor":"white","linewidth":2})
axes[0,0].set_title("Customer Segment Distribution")

# 2. RFM boxplots by segment
for ax, metric, idx in zip(
    [axes[0,1], axes[1,0], axes[1,1]],
    ["Recency","Frequency","Monetary"],
    [(0,1),(1,0),(1,1)]
):
    seg_order = list(SEG_COLORS.keys())
    data_boxes = [rfm_global[rfm_global["Segment"]==s][metric].values
                  for s in seg_order]
    bp = ax.boxplot(data_boxes,
                    labels=[s.replace(" ","\n") for s in seg_order],
                    patch_artist=True, notch=False)
    for patch, seg in zip(bp["boxes"], seg_order):
        patch.set_facecolor(SEG_COLORS[seg])
        patch.set_alpha(0.75)
    ax.set_title(f"{metric} by Segment")
    ax.set_ylabel(metric)

plt.suptitle("Customer Segment Analysis — RFM Profiles",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

# PCA 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(rfm_scaled)
fig, ax = plt.subplots(figsize=(9, 7))
for seg, color in SEG_COLORS.items():
    mask = rfm_global["Segment"] == seg
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color, label=seg, alpha=0.5, s=12, edgecolors="none")
ax.set_xlabel(f"PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("Customer Segments — PCA 2D Projection")
ax.legend(markerscale=3, fontsize=10)
plt.tight_layout(); plt.show()

## CELL 21 — Per-Country Segment Breakdown

In [ ]:
print("=" * 60)
print("PER-COUNTRY SEGMENT BREAKDOWN — Focus Countries")
print("=" * 60)

fig, axes = plt.subplots(1, len(FOCUS_COUNTRIES),
                          figsize=(4*len(FOCUS_COUNTRIES), 5))

for idx, country in enumerate(FOCUS_COUNTRIES):
    sub = rfm_global[rfm_global["Country"] == country]
    if sub.empty:
        axes[idx].set_title(f"{country}\n(no data)")
        continue
    seg_c = sub["Segment"].value_counts()
    colors_c = [SEG_COLORS.get(s, "#888") for s in seg_c.index]
    axes[idx].pie(seg_c.values, labels=seg_c.index,
                  colors=colors_c, autopct="%1.0f%%",
                  startangle=140,
                  wedgeprops={"edgecolor":"white","linewidth":1.5},
                  textprops={"fontsize":8})
    tier = country_profile[country_profile["Country"]==country]["Revenue_Tier"].values[0]
    axes[idx].set_title(f"{country}\n({tier} | {len(sub)} custs)", fontsize=9)

plt.suptitle("Customer Segment Distribution per Country",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()

# Table
print("\n  Per-Country Segment Counts:")
seg_country_pivot = (rfm_global[rfm_global["Country"].isin(FOCUS_COUNTRIES)]
                     .groupby(["Country","Segment"])
                     .size()
                     .unstack(fill_value=0))
display(seg_country_pivot)

## CELL 22 — ML Classification: Prepare Dataset

In [ ]:
print("=" * 60)
print("ML CLASSIFICATION — Predict Customer Segment")
print("=" * 60)

X = rfm_global[["Recency","Frequency","Monetary"]].copy()
y = rfm_global["Segment"].copy()

le = LabelEncoder()
y_enc = le.fit_transform(y)

print(f"  Feature matrix : {X.shape}")
print(f"  Target classes : {le.classes_}")
print(f"  Class distribution:\n{pd.Series(y).value_counts().to_string()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.20, random_state=42, stratify=y_enc
)
scaler_cls = StandardScaler()
X_train_sc = scaler_cls.fit_transform(X_train)
X_test_sc  = scaler_cls.transform(X_test)

print(f"\n  Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

## CELL 23 — Train 4 ML Classifiers & Compare

In [ ]:
print("=" * 60)
print("TRAINING 4 ML CLASSIFICATION MODELS")
print("=" * 60)

seg_names = list(le.classes_)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

ml_results = {}
predictions = {}

for name, model in models.items():
    use_scaled = (name == "Logistic Regression")
    Xtr = X_train_sc if use_scaled else X_train.values
    Xte = X_test_sc  if use_scaled else X_test.values

    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    cv_f1 = cross_val_score(model, Xtr, y_train,
                             cv=5, scoring="f1_weighted", n_jobs=-1).mean()
    ml_results[name] = {
        "Accuracy":  round(accuracy_score(y_test, preds),  4),
        "Precision": round(precision_score(y_test, preds, average="weighted", zero_division=0), 4),
        "Recall":    round(recall_score(y_test, preds, average="weighted",    zero_division=0), 4),
        "F1":        round(f1_score(y_test, preds, average="weighted",        zero_division=0), 4),
        "CV F1":     round(cv_f1, 4),
        "model":     model,
    }
    predictions[name] = preds

    print(f"\n  {name}")
    print(f"    Accuracy  : {ml_results[name]['Accuracy']:.4f}")
    print(f"    Precision : {ml_results[name]['Precision']:.4f}")
    print(f"    Recall    : {ml_results[name]['Recall']:.4f}")
    print(f"    F1 Score  : {ml_results[name]['F1']:.4f}")
    print(f"    CV F1     : {ml_results[name]['CV F1']:.4f}")

best_model_name = max(ml_results, key=lambda k: ml_results[k]["F1"])
print(f"\n  🏆 Best ML Model: {best_model_name} "
      f"(F1={ml_results[best_model_name]['F1']:.4f})")

## CELL 24 — Model Comparison Charts + Confusion Matrices

In [ ]:
# Comparison bar chart
model_names = list(ml_results.keys())
metrics     = ["Accuracy","Precision","Recall","F1"]
x_pos       = np.arange(len(model_names))
width       = 0.2

fig, ax = plt.subplots(figsize=(14, 5))
for i, metric in enumerate(metrics):
    vals = [ml_results[m][metric] for m in model_names]
    ax.bar(x_pos + i*width, vals, width,
           label=metric, color=PALETTE[i], alpha=0.85)

ax.set_xticks(x_pos + 1.5*width)
ax.set_xticklabels(model_names, rotation=10)
ax.set_ylim([0.8, 1.02])
ax.set_title("ML Classifier Comparison — All Metrics")
ax.set_ylabel("Score"); ax.legend()
plt.tight_layout(); plt.show()

# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()
for ax, (name, preds) in zip(axes, predictions.items()):
    use_scaled = (name == "Logistic Regression")
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=seg_names).plot(
        ax=ax, colorbar=False, cmap="Blues")
    acc = ml_results[name]["Accuracy"]
    ax.set_title(f"{name}\nAccuracy={acc:.4f}", fontsize=11)

plt.suptitle("Confusion Matrices — All 4 Classifiers",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

# Detailed classification reports
print("\nDetailed Classification Reports:")
for name, preds in predictions.items():
    print(f"\n  ── {name} ──")
    print(classification_report(y_test, preds,
                                target_names=seg_names,
                                zero_division=0))

## CELL 25 — Hyperparameter Tuning (Best Model)

In [ ]:
print("=" * 60)
print(f"HYPERPARAMETER TUNING — {best_model_name}")
print("=" * 60)

param_grids = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10, 100],
        "solver": ["lbfgs","saga"],
    },
    "Decision Tree": {
        "max_depth":         [None, 5, 10, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf":  [1, 2, 4],
    },
    "Random Forest": {
        "n_estimators":      [100, 200],
        "max_depth":         [None, 10, 20],
        "min_samples_split": [2, 5],
    },
    "Gradient Boosting": {
        "n_estimators":  [100, 200],
        "max_depth":     [3, 5, 7],
        "learning_rate": [0.05, 0.1, 0.2],
    },
}

best_model_obj = models[best_model_name]
pg             = param_grids[best_model_name]
use_scaled     = (best_model_name == "Logistic Regression")
Xtr = X_train_sc if use_scaled else X_train.values
Xte = X_test_sc  if use_scaled else X_test.values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gs = GridSearchCV(best_model_obj, pg,
                  cv=cv, scoring="f1_weighted",
                  n_jobs=-1, verbose=0)
gs.fit(Xtr, y_train)

final_model  = gs.best_estimator_
tuned_preds  = final_model.predict(Xte)
tuned_f1     = f1_score(y_test, tuned_preds, average="weighted", zero_division=0)
tuned_acc    = accuracy_score(y_test, tuned_preds)

print(f"\n  Best parameters : {gs.best_params_}")
print(f"  Tuned F1 Score  : {tuned_f1:.4f}")
print(f"  Tuned Accuracy  : {tuned_acc:.4f}")
print(f"  Improvement     : +{(tuned_f1 - ml_results[best_model_name]['F1']):.4f} F1")

## CELL 26 — Feature Importance

In [ ]:
print("=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)

fi_model, fi_name = None, None
for name in ["Random Forest","Gradient Boosting","Decision Tree"]:
    if hasattr(models[name], "feature_importances_"):
        fi_model = models[name]; fi_name = name; break

if fi_model:
    fi = pd.Series(fi_model.feature_importances_,
                   index=["Recency","Frequency","Monetary"]).sort_values()
    fig, ax = plt.subplots(figsize=(8, 4))
    colors_fi = [PALETTE[0] if v > fi.mean() else "#94A3B8"
                 for v in fi.values]
    ax.barh(fi.index, fi.values, color=colors_fi)
    ax.set_title(f"Feature Importance — {fi_name}")
    ax.set_xlabel("Importance Score")
    for i, v in enumerate(fi.values):
        ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=11)
    plt.tight_layout(); plt.show()

    print("\n  Feature Rankings:")
    for feat, imp in fi.sort_values(ascending=False).items():
        print(f"    {feat:<15}: {imp:.4f}")

## CELL 27 — Business Insights & Recommendations

In [ ]:
print("=" * 60)
print("BUSINESS INSIGHTS & RECOMMENDATIONS")
print("=" * 60)

total_rev   = df["TotalAmount"].sum()
top_country_name = df.groupby("Country")["TotalAmount"].sum().idxmax()

champions  = rfm_global[rfm_global["Segment"]=="Champions"]
loyal      = rfm_global[rfm_global["Segment"]=="Loyal Customers"]
at_risk    = rfm_global[rfm_global["Segment"]=="At-Risk Customers"]
new_occ    = rfm_global[rfm_global["Segment"]=="New / Occasional"]

print(f"""
  ┌─────────────────────────────────────────────────────────┐
  │              KEY BUSINESS METRICS                       │
  ├─────────────────────────────────────────────────────────┤
  │  Total Revenue       : £{total_rev:>12,.2f}               │
  │  Unique Customers    : {len(rfm_global):>12,}               │
  │  Top Country         : {top_country_name:<28}   │
  │  Total Countries     : {df['Country'].nunique():>12,}               │
  └─────────────────────────────────────────────────────────┘
""")

print("  CUSTOMER SEGMENTS:")
for seg_name, seg_df in [
    ("Champions",        champions),
    ("Loyal Customers",  loyal),
    ("At-Risk Customers",at_risk),
    ("New / Occasional", new_occ),
]:
    pct = len(seg_df)/len(rfm_global)*100
    print(f"\n  [{seg_name}] — {len(seg_df):,} customers ({pct:.1f}%)")
    print(f"    Avg Recency  : {seg_df['Recency'].mean():.0f} days")
    print(f"    Avg Frequency: {seg_df['Frequency'].mean():.1f} orders")
    print(f"    Avg Monetary : £{seg_df['Monetary'].mean():,.2f}")

print("""
  ── KEY INSIGHTS ────────────────────────────────────────────
  1. UK dominates (85%+ revenue) — EU expansion is the
     highest growth opportunity.
  2. Champions & Loyal (~50%) drive majority of revenue —
     focus retention and upsell programs here.
  3. At-Risk segment (~25%) needs win-back campaigns with
     personalised discounts within 30 days.
  4. Mid-week (Tue–Thu), 10AM–2PM is the peak ordering window —
     time promotions and emails accordingly.
  5. SARIMA forecasts show Nov–Dec seasonal revenue spikes
     (+40%) — stock up by October.
""")

## CELL 28 — Final Summary Table

In [ ]:
# Rebuild common from successful fits
common = [c for c in FOCUS_COUNTRIES if c in arima_results and c in sarima_results]

print("\n" + "=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)

print("\n── Time Series Models (per Country) ───────────────────────────────")
print(f"  {'Country':<26}| {'ARIMA Order':>12} | {'ARIMA R²':>9} | {'SARIMA R²':>9} | Winner")
print("  " + "-"*65)
for c in common:
    a = arima_results[c]; s = sarima_results[c]
    winner = "SARIMA ✓" if s["rmse"] < a["rmse"] else "ARIMA  ✓"
    print(f"  {c:<26}| {str(a['order']):>12} | {a['r2']:>9.4f} | {s['r2']:>9.4f} | {winner}")

print("\n── ML Classification Models ────────────────────────────────────────")
print(f"  {'Model':<22} | {'Accuracy':>9} | {'F1':>9} | {'CV F1':>9}")
print("  " + "-"*55)
for name in ml_results:
    r = ml_results[name]
    marker = " ◄ BEST" if name == best_model_name else ""
    print(f"  {name:<22} | {r['Accuracy']:>9.4f} | {r['F1']:>9.4f} | {r['CV F1']:>9.4f}{marker}")
print(f"\n  Tuned {best_model_name}: Accuracy={tuned_acc:.4f}  F1={tuned_f1:.4f}")

print("\n── Clustering ──────────────────────────────────────────────────────")
print(f"  Algorithm      : KMeans (k={BEST_K})")
print(f"  Silhouette     : {sil_final:.4f}")
print(f"  Davies-Bouldin : {dbi_final:.4f}")
print(f"  Segments       : {list(SEG_COLORS.keys())}")

print("\n✅  All done! Scroll up to see all plots, tables, and forecasts.")

## CELL 29 — Save Models & Download

In [ ]:
os.makedirs("models", exist_ok=True)

pipeline = {
    "model":           final_model,
    "model_name":      best_model_name,
    "scaler_rfm":      scaler_rfm,
    "scaler_cls":      scaler_cls,
    "kmeans":          kmeans,
    "label_encoder":   le,
    "segment_labels":  _label_map,
    "best_k":          BEST_K,
    "silhouette":      sil_final,
    "feature_names":   ["Recency","Frequency","Monetary"],
}
joblib.dump(pipeline, "models/pipeline.pkl")

# Save forecasts JSON
with open("models/country_forecasts.json","w") as f:
    json.dump(country_forecasts, f, indent=2)

# Save RFM + segments CSV
rfm_global.to_csv("models/rfm_segments.csv", index=False)

print("✅  Models saved:")
print("   models/pipeline.pkl")
print("   models/country_forecasts.json")
print("   models/rfm_segments.csv")

# Optional: Download all files
# from google.colab import files
# for fname in ["models/pipeline.pkl","models/rfm_segments.csv","models/country_forecasts.json"]:
#     files.download(fname)

## CELL 30 — Inference Demo

In [ ]:
print("=" * 60)
print("INFERENCE DEMO — Predict Segment for New Customer")
print("=" * 60)

saved = joblib.load("models/pipeline.pkl")
_model      = saved["model"]
_scaler_cls = saved["scaler_cls"]
_le         = saved["label_encoder"]
_use_scaled = (saved["model_name"] == "Logistic Regression")

def predict_segment(recency: float, frequency: float, monetary: float) -> dict:
    """Predict RFM segment for a single customer."""
    X_new = pd.DataFrame([[recency, frequency, monetary]],
                          columns=["Recency","Frequency","Monetary"])
    if _use_scaled:
        X_new = _scaler_cls.transform(X_new)
    cluster  = int(_model.predict(X_new)[0])
    label    = _le.inverse_transform([cluster])[0]
    proba    = _model.predict_proba(X_new)[0] if hasattr(_model,"predict_proba") else None
    return {"segment": label, "probabilities": proba}

# Test cases
test_cases = [
    (5,  45, 8500, "High-value recent buyer"),
    (90,  3,  200, "Lapsed low-value buyer"),
    (15, 20, 2500, "Regular mid-value buyer"),
    (200, 1,   50, "One-time old buyer"),
]
print(f"\n  {'Customer Profile':<30} | {'Segment':<20}")
print("  " + "-"*55)
for rec, freq, mon, desc in test_cases:
    result = predict_segment(rec, freq, mon)
    print(f"  R={rec:>3} F={freq:>2} M=£{mon:>5} ({desc:<20}) → {result['segment']}")